# OpenAI API (gpt-4o-mini) — 3-Condition RACE Experiment

## Mutual Awareness in Semantic Communication: blind / choices_aware / full_aware

Replicates `qwen_agent_env.ipynb` Key Idea 1 using OpenAI API instead of local Qwen3-4B.

- **A (Summarizer)**: Reads passage, writes concise summary under token budget
- **B (Answerer)**: Receives summary + question + choices, outputs answer (never sees passage)
- 3 conditions vary what A sees: passage only / + choices / + question & choices
- Token budgets: 16 / 32 / 48 / 64
- 30 RACE-high questions, seed=42, temperature=0

### Conditions

| Condition | A sees | Info level |
|-----------|--------|-----------|
| blind | passage only | none |
| choices_aware | passage + choices | medium |
| full_aware | passage + question + choices | high |

In [ ]:
# ============================================================
# Cell 1: Setup — OpenAI API + RACE loader
# ============================================================
!pip install -q openai datasets

import os
import random
import re
from openai import OpenAI
from datasets import load_dataset

# API Key (set in Colab or environment)
# os.environ["OPENAI_API_KEY"] = "sk-..."  # uncomment and fill
client = OpenAI()

MODEL = "gpt-4o-mini"

def chat(system_prompt, user_message, max_tokens=256):
    """OpenAI API call wrapper."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
        max_tokens=max_tokens,
        temperature=0,
    )
    text = response.choices[0].message.content.strip()
    tokens = response.usage.completion_tokens
    return {"response": text, "tokens": tokens}

# RACE loader
print("Loading RACE...")
race = load_dataset("ehovy/race", "high", split="test")

def format_race(ex):
    choices = ex["options"]
    choices_text = "\n".join(f"{chr(65+i)}) {c}" for i, c in enumerate(choices))
    return {
        "passage": ex["article"],
        "question": ex["question"],
        "choices_text": choices_text,
        "answer": ex["answer"],
    }

def sample_race(n, seed=42):
    random.seed(seed)
    samples = random.sample(list(race), min(n, len(race)))
    return [format_race(s) for s in samples]

def extract_answer(response):
    text = response.strip()
    if text.upper() in ("A", "B", "C", "D"):
        return text.upper()
    m = re.search(r'\\boxed\{([A-D])\}', text)
    if m: return m.group(1)
    m = re.search(r'(?:answer|choice)[\s:is]*([A-D])\b', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    m = re.search(r'^([A-D])[)\.]', text, re.MULTILINE)
    if m: return m.group(1)
    m = re.search(r'\b([A-D])\b', text)
    if m: return m.group(1)
    return "N/A"

print(f"RACE loaded: {len(race)} questions")
print(f"Model: {MODEL}")

In [ ]:
# ============================================================
# Cell 2: 3-Condition Experiment (blind / choices_aware / full_aware)
# RACE 30 questions x 4 token budgets x 3 conditions
# OpenAI API (gpt-4o-mini, temperature=0)
# ============================================================

# --- A prompts ---
A_BLIND = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. Start with the most important fact. "
    "Capture the most important facts and events."
)

A_CHOICES = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader must choose between the answer options shown below. "
    "Start with the ONE fact that best distinguishes between the options. "
    "Do NOT indicate which option is correct."
)

A_FULL = (
    "You are a Summarizer. Read the passage and write a concise summary "
    "in 2-3 sentences. The reader needs to answer the question shown below. "
    "Start with the fact most relevant to the question. "
    "Do NOT state the answer directly."
)

# --- B prompt (same for all conditions) ---
B_PROMPT = (
    "You are an Answerer. You receive a summary of a passage. "
    "Use it to answer the question. "
    "Output ONLY a single letter: A, B, C, or D."
)

# --- 3 conditions ---
CONDITIONS = ["blind", "choices_aware", "full_aware"]

TX_BUDGETS = [16, 32, 48, 64]

# Sample 30 RACE questions
Q1 = sample_race(30, seed=42)
print(f"Sampled {len(Q1)} RACE questions (seed=42)")
for q in Q1[:3]:
    print(f"  Q: {q['question'][:50]}... ans={q['answer']}")

def run_3cond(questions):
    results = {b: {c: [] for c in CONDITIONS} for b in TX_BUDGETS}

    for qi, q in enumerate(questions):
        print(f"\n--- Q{qi+1}/{len(questions)}: {q['question'][:60]}...")
        print(f"    Expected: {q['answer']}")

        for budget in TX_BUDGETS:
            for cond in CONDITIONS:
                # --- A (Summarizer) ---
                if cond == "blind":
                    a_prompt = A_BLIND
                    a_input = f"Passage:\n{q['passage']}"
                elif cond == "choices_aware":
                    a_prompt = A_CHOICES
                    a_input = f"Passage:\n{q['passage']}\n\nAnswer options:\n{q['choices_text']}"
                else:  # full_aware
                    a_prompt = A_FULL
                    a_input = (
                        f"Passage:\n{q['passage']}\n\n"
                        f"Question: {q['question']}\nOptions:\n{q['choices_text']}"
                    )

                a_out = chat(a_prompt, a_input, max_tokens=budget)

                # --- B (Answerer) ---
                b_msg = (
                    f"Summary:\n{a_out['response']}\n\n"
                    f"Question: {q['question']}\nChoices:\n{q['choices_text']}\n\nAnswer:"
                )
                b_out = chat(B_PROMPT, b_msg, max_tokens=48)
                final_ans = extract_answer(b_out["response"])
                correct = final_ans == q["answer"]

                results[budget][cond].append({
                    "correct": correct,
                    "answer": final_ans,
                    "expected": q["answer"],
                    "a_tokens": a_out["tokens"],
                })

        # Log @lowest budget
        b = TX_BUDGETS[0]
        row = "  ".join(
            f"{c}: {results[b][c][-1]['answer']}"
            f"{'✓' if results[b][c][-1]['correct'] else '✗'}"
            for c in CONDITIONS
        )
        print(f"  @{b}tok: {row}")

    return results

print(f"\nRunning 3-condition experiment (gpt-4o-mini, temp=0)...")
print(f"Budgets: {TX_BUDGETS}, Questions: {len(Q1)}, Conditions: {CONDITIONS}")
print(f"Total API calls: {len(Q1) * len(TX_BUDGETS) * len(CONDITIONS) * 2} (A+B per trial)")
k1_results = run_3cond(Q1)

# ============================================================
# Results table
# ============================================================
print(f"\n{'='*50}")
print(f"  gpt-4o-mini — 3 Conditions (RACE-high, n={len(Q1)})")
print(f"{'='*50}")
print(f"\n{'Budget':<8} {'blind':<12} {'choices':<12} {'full':<12}")
print("-" * 44)
for budget in TX_BUDGETS:
    row = f"{budget}tok  "
    for cond in CONDITIONS:
        res = k1_results[budget][cond]
        n = len(res)
        acc = sum(r["correct"] for r in res) / n
        row += f"{acc*100:>5.0f}%      "
    print(row)

# Rate-distortion curve
print(f"\n--- Rate-Distortion Curve ---")
for cond in CONDITIONS:
    points = " -> ".join(
        f"{b}({sum(r['correct'] for r in k1_results[b][cond])/len(k1_results[b][cond])*100:.0f}%)"
        for b in TX_BUDGETS
    )
    print(f"  {cond:<16}: {points}")

# Key findings
print(f"\n--- Key Findings ---")
for budget in TX_BUDGETS:
    accs = {c: sum(r["correct"] for r in k1_results[budget][c]) / len(k1_results[budget][c]) for c in CONDITIONS}
    print(f"  @{budget}tok: blind {accs['blind']*100:.0f}% -> choices {accs['choices_aware']*100:.0f}% -> full {accs['full_aware']*100:.0f}%")

# Average A tokens actually used (API can stop early)
print(f"\n--- Avg A tokens used (API early stopping) ---")
for budget in TX_BUDGETS:
    for cond in CONDITIONS:
        res = k1_results[budget][cond]
        avg_tok = sum(r["a_tokens"] for r in res) / len(res)
        print(f"  @{budget}tok {cond:<16}: avg {avg_tok:.1f} tokens used")

print(f"\n--- Compare with Qwen3-4B local results ---")
print("Note: gpt-4o-mini uses proper max_tokens (model can stop early),")
print("while Qwen3-4B local uses greedy decoding (always generates max tokens).")
print("This may lead to different rate-distortion characteristics.")